In [ ]:
import pandas as pd
import lightgbm as lgb
import numpy as np
import matplotlib.pyplot as plt


from sklearn.model_selection import train_test_split

from data_loader import split_data

In [ ]:
path = r'data\train.csv'

In [ ]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9
}

Base rmse = 0.130486 on train set

List of features to drop is below

In [ ]:
lst_1 = ['RoofMatl', 'HouseStyle', 'LandContour', 'LandSlope', 'Alley', 'BldgType', 'Street', 'MSZoning', 'RoofStyle', 'LowQualFinSF', 'Heating', 'BsmtFinType2', 'Utilities', 'Condition2', 'HeatingQC', 'Electrical']

lst_2 = ['RoofMatl', 'HouseStyle', 'LandContour', 'LandSlope', 'Alley', 'BldgType', 'Street', 'MSZoning', 'RoofStyle', 'LowQualFinSF', 'Heating', 'BsmtFinType2', 'Utilities', 'Condition2', 'HeatingQC', 'Electrical', 'PoolArea']

In [ ]:
num_round=10000

train_data, test_data = split_data(path, stage='lgb', cols_drop=lst_1)

bst = lgb.train(params, train_data, num_round,valid_sets=[test_data], callbacks=[lgb.early_stopping(stopping_rounds=10)])

In [ ]:
from data_loader import test_data_func

test_path = r'data\test.csv'

df_test, cat_cols = test_data_func(test_path, cols_drop=lst_1)

In [ ]:
ids = df_test['Id'].to_list()

predictions = bst.predict(df_test.drop(['Id'], axis=1))
predictions = np.expm1(predictions)

result = pd.DataFrame({'Id':ids, 'SalePrice':predictions})

result.head(5)

In [ ]:
result.to_csv('results.csv', index=False)

In [ ]:
important_features=['GrLivArea', 'LotArea', '1stFlrSF', 'TotalBsmtSF', 'BsmtFinSF1', 'GarageArea', 'OpenPorchSF', 'LotFrontage']

In [ ]:
fig, ax = plt.subplots(figsize=(12,12))
lgb.plot_importance(bst, ax=ax)

In [ ]:
# bst.save_model('model.txt', num_iteration=bst.best_iteration)

##### Trying to remove some unimportant features and looking at rmse

In [ ]:
# features_df = pd.DataFrame({'Importance':bst.feature_importance(), 'Feature name':bst.feature_name()}).sort_values(['Importance'], ascending=False)
# df_types = pd.DataFrame(X_train[bst.feature_name()].dtypes, columns=['Type']).reset_index(drop=False)
# features_df=features_df.reset_index(drop=True)

# df_types = pd.DataFrame(X_train[bst.feature_name()].dtypes, columns=['Type']).reset_index(drop=False)
# df_types = df_types.rename({'index':'Feature name'}, axis=1)

# df_features = pd.merge(features_df, df_types, on='Feature name') \
#                 .sort_values('Importance', ascending=True) \
#                 .reset_index(drop=True)

# features_to_remove = df_features['Feature name'].to_list() # 79 features

##### Above code is made to create this list of features with ascending importance 

In [ ]:
features_to_remove = ['RoofMatl', 'HouseStyle', 'LandContour', 'LandSlope', 'Alley', 'BldgType', 'Street', 'MSZoning', 'RoofStyle', 'LowQualFinSF', 'Heating', 'BsmtFinType2', 'Utilities', 'Condition2', 'HeatingQC', 'Electrical', 'PoolArea', 'SaleCondition', 'SaleType', 'MiscFeature', 'MiscVal', 'Fence', 'PoolQC', '3SsnPorch', 'GarageQual', 'GarageCond', 'BsmtHalfBath', 'GarageType', 'Functional', 'LotConfig', 'PavedDrive', 'BsmtFinSF2', 'Foundation', 'Exterior2nd', 'KitchenAbvGr', 'BsmtQual', 'KitchenQual', 'EnclosedPorch', 'Condition1', 'MasVnrType', 'ExterCond', 'FireplaceQu', 'LotShape', 'Exterior1st', 'BsmtExposure', 'FullBath', 'BsmtCond', 'HalfBath', 'ScreenPorch', 'CentralAir', 'BsmtFinType1', 'BsmtFullBath', 'GarageCars', 'BedroomAbvGr', 'ExterQual', 'GarageFinish', 'TotRmsAbvGrd', 'YrSold', 'MSSubClass', 'Fireplaces', 'MasVnrArea', 'WoodDeckSF', 'MoSold', '2ndFlrSF', 'BsmtUnfSF', 'Neighborhood', 'OpenPorchSF', 'OverallQual', 'LotFrontage', 'OverallCond', 'YearBuilt', 'GarageYrBlt', 'YearRemodAdd', 'GarageArea', 'BsmtFinSF1', '1stFlrSF', 'TotalBsmtSF', 'LotArea', 'GrLivArea']

In [ ]:
cat_cols = ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition']

In [ ]:
dp_lst = []

data_path = r'data\train.csv'

best_score_dct = {}

for ind, f in enumerate(features_to_remove[:62]):

    dp_lst.append(f)

    train_data, val_data = split_data(data_path, cols_drop=dp_lst)

    num_round=10000
    bst = lgb.train(params, train_data, num_round,valid_sets=[val_data], callbacks=[lgb.early_stopping(stopping_rounds=10)])
    best_score_dct[ind] = bst.best_score['valid_0']['rmse']

    if (ind == 15) or (ind == 16):
        print(dp_lst)
    


In [ ]:
# Extract keys (index) and values
indices = list(best_score_dct.keys())
values = list(best_score_dct.values())

# Create line plot
plt.figure(figsize=(10, 4.5))
plt.plot(
    indices,
    values,
    marker='o',
    linestyle='-',
    color='#1f77b4',
    linewidth=2,
    markersize=8,
)

plt.plot((0, 61),(0.130486, 0.130486), color='red', linestyle='-')

# Customize grid, labels, and ticks
plt.title('Metric Value vs. Index', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Index', fontsize=11)
plt.ylabel('Metric Value', fontsize=11)
plt.xticks(indices, fontsize=7)
plt.grid(True, linestyle='--', alpha=0.6)


y_min, y_max = min(values), max(values)
plt.ylim(y_min - (y_max - y_min) * 0.3, y_max + (y_max - y_min) * 0.4)

plt.tight_layout()
plt.show()

##### Feature split that shows better rmse is following:

These columns need to be dropped :)